In [37]:
import os
import yaml
import pandas as pd
import numpy as np

with open("../../config.local.yaml", "r") as f:
    LOCAL_CONFIG = yaml.safe_load(f)

LOCAL_PATH = LOCAL_CONFIG["LOCAL_PATH"]
DATA_PATH = LOCAL_CONFIG["DATA_PATH"]

ANALYSIS_FILENAME = "tax_analysis_panel.parquet"


In [38]:
regs_df = pd.read_csv(os.path.join(DATA_PATH, "best_treatment_dates.csv"))
tax_df = pd.read_excel(os.path.join(DATA_PATH, "FiSC-Full-Dataset-2023-Update.xlsx"), sheet_name="Data")

In [39]:
# clean dates
regs_df['best_enforcement'] = pd.to_datetime(regs_df['best_enforcement'], errors='coerce')
regs_df['best_passage'] = pd.to_datetime(regs_df['best_passage'], errors='coerce')# clean city names

In [40]:
# Clean city names
mask = tax_df['city_name'] == "TX: Ft. Worth"
tax_df.loc[mask, 'city_name'] = "TX: Fort Worth"

mask = tax_df['city_name'] == "OK: Oklahoma"
tax_df.loc[mask, 'city_name'] = "OK: Oklahoma City"

regs_df['city_name'] = regs_df['state'] + ": " + regs_df['city']

In [41]:
# check match quality

regs_cities = set(regs_df['city_name'].unique())
tax_cities = set(tax_df['city_name'].unique())

assert regs_cities.issubset(tax_cities)


In [42]:
# make the merge

tax_cols = [
    'city_name', 'year',
    'rev_general',
    'taxes', 'tax_property', 'tax_sales_general', 'tax_income', 'tax_other', 'tax_licenses'
]

df = regs_df.merge(tax_df[tax_cols], on='city_name', how='left')


In [43]:
# time measures
df['enforcement_year'] = df['best_enforcement'].dt.year
df['passage_year'] = df['best_passage'].dt.year
df['years_from_enforcement'] = (df['year'] - df['enforcement_year'])
df['years_from_passage'] = (df['year'] - df['passage_year'])

In [44]:
# for cities with an enforcement date, drop rows >6 years to/from enforcement
mask = (np.abs(df['years_from_enforcement']) > 6) & (df['years_from_enforcement'].notna())
df = df.loc[~mask].reset_index(drop=True)

# for cities without an enforcement date, drop years outside the min/max of the remaining data
year_min = df.loc[df['best_enforcement'].notna(), 'year'].min()
year_max = df.loc[df['best_enforcement'].notna(), 'year'].max()
mask = (df['year'] < year_min) | (df['year'] > year_max)
df = df.loc[~mask].reset_index(drop=True)


In [45]:
# change enforcement and passage year to 0 for cities without enforcement/passage dates
df.loc[df['best_enforcement'].isna(), 'enforcement_year'] = 0
df.loc[df['best_passage'].isna(), 'passage_year'] = 0
df['enforcement_year'] = df['enforcement_year'].astype(int)
df['passage_year'] = df['passage_year'].astype(int)


In [46]:
# make a city_id integer
df['city_id'] = df['city_name'].astype('category').cat.codes

In [47]:
# output dataframe for analysis
df.to_parquet(os.path.join(DATA_PATH, ANALYSIS_FILENAME))

In [48]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 572 entries, 0 to 571
Data columns (total 21 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   city                    572 non-null    str           
 1   state                   572 non-null    str           
 2   best_passage            500 non-null    datetime64[us]
 3   best_passage_src        572 non-null    str           
 4   best_enforcement        482 non-null    datetime64[us]
 5   best_enforcement_src    572 non-null    str           
 6   confidence              572 non-null    str           
 7   city_name               572 non-null    str           
 8   year                    572 non-null    int64         
 9   rev_general             572 non-null    float64       
 10  taxes                   572 non-null    float64       
 11  tax_property            572 non-null    float64       
 12  tax_sales_general       572 non-null    float64       
 13  t